In [ ]:
#| hide

# Project Setup

> How this project was initialised and how to get started as a new developer.

## 1. Quick Start

If you cloned this repository, you only need **one command** to be up and running.

**Prerequisites:** Python ≥ 3.10, git, a terminal and [uv](https://docs.astral.sh/uv/).

```sh
# Clone the repo
$ git clone https://github.com/Ramon-PR/ros2_cam_nbdev.git
$ cd ros2_cam_nbdev

# Create the virtual environment, install deps, and link the package
$ uv sync

# Export notebooks to Python modules (creates ros2_cam_nbdev/*.py)
$ uv run nbdev-export
```

That's it! You only need to run `uv sync` again when:
- You clone the repo on a new machine
- Dependencies change (`pyproject.toml` or `uv.lock` updates)

Verify the environment is working:

```sh
$ uv run python -c "import ros2_cam_nbdev; print('OK')"
OK
```

> **Note:** If `uv` is not installed, the [official Astral installer](https://docs.astral.sh/uv/getting-started/installation/) handles it:
> ```sh
> $ curl -LsSf https://astral.sh/uv/install.sh | sh
> ```

## 2. What `uv sync` Does

`uv sync` is a single, reproducible step:

1. Reads `pyproject.toml` for dependency declarations.
2. Reads `uv.lock` for the exact pinned versions — **no surprises, no floating deps**.
3. Creates `.venv/` using `--system-site-packages` so that ROS2 Python packages (e.g. `rclpy`) installed system-wide are accessible inside the virtual environment.
4. Installs all dependencies at the pinned versions.
5. **Links the local package in editable mode** — creates a `.pth` file pointing to your project directory.

The result is a fully reproducible environment on any machine that has the same base ROS2 install.

You only need to run `uv sync` again when:
- Cloning the repo on a new machine
- Dependencies change (`pyproject.toml` or `uv.lock` updates)

It is NOT needed after every `nbdev-export` — the editable link already points to your project directory.

## 3. How This Project Was Originally Bootstrapped

> **For reference only.** You do NOT need to run these steps — the repo is already initialised.

This project was created from the [template_uv_nbdev](https://github.com/Ramon-PR/template_uv_nbdev) template. The original setup required two scripts:

| Script | Shell | What it does |
|--------|-------|--------------|
| `scripts/basic_install.sh` | bash | Detects the OS package manager (apt / dnf / brew / …) and installs `curl`, `git`, `python3`, and `uv` |
| `scripts/prepare_project.bash` | bash | Creates `.venv` with `--system-site-packages`, pins `matplotlib-inline==0.1.6`, runs `nbdev-new`, and adds `ipykernel` |

Full documentation of the template is in [`README_uv_nbdev.md`](../README_uv_nbdev.md).
The original ROS2 development template README is at [`README_template_ros2dev.md`](../README_template_ros2dev.md).

## 4. Project Structure

```
ros2_cam_nbdev/
├── nbs/                        # ✏️  SOURCE OF TRUTH — edit notebooks here
│   ├── index.ipynb             #    Home page / README source
│   ├── 00_project_setup.ipynb  #    This file — project setup guide
│   └── ...                     #    Other notebooks for code + docs
│   └── nbdev.yml               #    Quarto / docs site configuration
│
├── ros2_cam_nbdev/             # ⚙️  AUTO-GENERATED Python package — DO NOT edit
│   └── __init__.py             #    Package init (empty)
│   └── *.py                    #    Exported from nbs/ by nbdev-export
│
├── scripts/                    # 🛠️  Helper scripts for setup, build, and ROS2
│   ├── basic_install.sh        #    System + uv installer (bash)
│   ├── prepare_project.bash    #    uv venv + nbdev scaffolding (bash)
│   ├── compile_pkg.sh          #    Generates setup.py + colcon build (bash)
│   ├── generate_ros2_ws.zsh    #    Creates ROS2 workspace (zsh)
│   ├── mk_ros2pkg_opencv.zsh   #    Creates ROS2 package with OpenCV deps (zsh)
│   ├── mk_ros2_pkg.sh          #    Creates ROS2 package (⚠️ zsh despite .sh)
│   └── run_cam.zsh             #    Checks camera, builds, runs ROS2 node (zsh)
│
├── pyproject.toml              # 📋  Project metadata + nbdev config + dependencies
├── uv.lock                     # 🔒  Pinned dependency lockfile
├── system_constraints.txt      # 🚫  Constraints for ROS2-conflicting packages
└── .venv/                      # 🐍  Virtual environment (gitignored)
```

> ⚠️ `scripts/mk_ros2_pkg.sh` has a **misleading `.sh` extension** — its shebang is `#!/bin/zsh`. Always check the shebang before running a script.

## 5. Key Conventions

### Notebooks are the source of truth

All Python source code lives in `nbs/` as Jupyter notebooks. The `ros2_cam_nbdev/*.py` files are **auto-generated** by `nbdev-export` — editing them directly will cause your changes to be overwritten.

| Do this | Not this |
|---------|----------|
| Edit `nbs/01_camera.ipynb` | Edit `ros2_cam_nbdev/camera.py` |

If you ever *must* edit a `.py` file directly (escape hatch), sync changes back with `uv run nbdev-update`.

### Always use `uv run`

All nbdev commands are run through `uv run` to ensure the correct virtual environment is used:

```sh
# ✅ Correct
$ uv run nbdev-export

# ❌ Wrong — bypasses .venv
$ nbdev-export
```

### Pre-commit ritual

Before every `git commit` that touches notebooks or exported code:

```sh
$ uv run nbdev-prepare
```

This runs `nbdev-export` + `nbdev-test` + `nbdev-clean` + `nbdev-readme` in one step.

### `.venv` uses `--system-site-packages`

The virtual environment is created with access to system-level Python packages. This is intentional — it exposes ROS2 packages like `rclpy`, `sensor_msgs`, and `cv_bridge` to the notebooks without reinstalling them.

### `matplotlib-inline==0.1.6` is pinned

`matplotlib-inline` is pinned to version `0.1.6` for ROS2 compatibility. **Do not upgrade it.** If you need to add a package that depends on a newer version, use the constraints file:

```sh
$ uv add <package> --constraint system_constraints.txt
```

## 6. Daily Workflow

| Task | Command |
|------|---------|
| Sync environment (first time / after deps changes) | `uv sync` |
| Export notebooks → modules | `uv run nbdev-export` |
| Run all tests | `uv run nbdev-test` |
| Test a single notebook | `uv run nbdev-test --path nbs/XX_name.ipynb` |
| Pre-commit (export + test + clean + readme) | `uv run nbdev-prepare` |
| Preview documentation site | `uv run nbdev-preview` |
| Clean notebook metadata | `uv run nbdev-clean` |
| Add a dependency | `uv add <package>` |
| Sync `.py` edits back to notebooks | `uv run nbdev-update` |

## 7. ROS2 Integration

This project is designed to develop and distribute ROS2 camera nodes. The ROS2-specific workflow will be covered in detail in later notebooks (Phase 1+). Here is a brief overview:

- **ROS2 distro**: detected at runtime via the `$ROS_DISTRO` environment variable — never hardcoded.
- **System packages**: `.venv` is created with `--system-site-packages`, so `rclpy`, `sensor_msgs`, `cv_bridge`, and other ROS2 Python packages are available inside the environment without reinstalling.
- **Build**: `bash scripts/compile_pkg.sh` generates `setup.py` from the template and then runs `colcon build`.
- **Run**: After sourcing the workspace overlay, nodes are launched with `ros2 run <package> <node>`.
- **Helper scripts** in `scripts/` cover workspace creation, package scaffolding, and node launch — see [Section 4](#4-project-structure) for the full list.

> More details in upcoming notebooks.

## 8. Troubleshooting

### `uv: command not found`

`uv` is not on your `PATH`. Install it and reload your shell:

```sh
$ curl -LsSf https://astral.sh/uv/install.sh | sh
$ source ~/.bashrc   # or ~/.zshrc
```

### `ModuleNotFoundError: No module named 'rclpy'`

ROS2 is not sourced in the current shell. Source your ROS2 installation first:

```sh
$ source /opt/ros/$ROS_DISTRO/setup.bash
$ uv sync   # recreate .venv so it picks up the newly sourced packages
```

> Make sure to source ROS2 **before** running `uv sync` so the system-site-packages are visible when the venv is created.

### `ModuleNotFoundError: No module named 'ros2_cam_nbdev'`

The package hasn't been exported yet. Run:

```sh
$ uv run nbdev-export    # creates ros2_cam_nbdev/*.py from notebooks
```

If you cloned the repo fresh, also run `uv sync` first.